In [6]:
%pip install transformers datasets torch scikit-learn tqdm


Note: you may need to restart the kernel to use updated packages.


In [7]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-multilingual-cased")
model = AutoModelForMaskedLM.from_pretrained("google-bert/bert-base-multilingual-cased")

Some weights of the model checkpoint at google-bert/bert-base-multilingual-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load dữ liệu
file_path = "./dataset/Amazon_Product_Review_cleaned.csv"
df = pd.read_csv(file_path)

# Chọn cột cần thiết & bỏ giá trị rỗng
df = df[['review_headline', 'star_rating']].dropna()

# Chuyển đổi nhãn về dạng số nguyên
df['star_rating'] = df['star_rating'].astype(int)

# Chia dữ liệu thành train & val (80-20)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["review_headline"].tolist(), df["star_rating"].tolist(), test_size=0.2, random_state=42
)


In [16]:
import torch
from torch.utils.data import Dataset, DataLoader

# Tokenizer max_length = 128 (tối ưu cho 3060)
class StarRatingDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_length)
        self.labels = [label - 1 for label in labels]  # 🔥 Chuyển từ [1-5] → [0-4]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)  # 🔥 Đảm bảo kiểu long
        return item

# Tạo dataset
train_dataset = StarRatingDataset(train_texts, train_labels, tokenizer)
val_dataset = StarRatingDataset(val_texts, val_labels, tokenizer)

# Batch size = 16 (phù hợp với RTX 3060)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


In [10]:
# # Create a subset of the train dataset with the first 1000 samples
# small_train_dataset = StarRatingDataset(
#     train_texts[:5000], train_labels[:5000], tokenizer
# )

# # Create a DataLoader for the subset
# small_train_loader = torch.utils.data.DataLoader(
#     small_train_dataset,  # Chỉ lấy 1000 mẫu để train nhanh hơn
#     batch_size=8,
#     shuffle=True
# )

# from torch.utils.data import Subset

# # Create a subset of the validation dataset with the first 500 samples
# small_val_dataset = Subset(val_dataset, list(range(2000)))

# # Create a DataLoader for the subset
# small_val_loader = torch.utils.data.DataLoader(
#     small_val_dataset,  # Chỉ lấy 500 mẫu validation
#     batch_size=4,  # Giảm batch_size để tính nhanh
#     shuffle=False
# )




In [28]:
import torch
import torch.cuda.amp as amp
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
from transformers import AutoModelForSequenceClassification, get_scheduler
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model initialization with correct number of labels (0-4)
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased", 
    num_labels=5  # For classes 0-4
).to(device)

# Torch optimization
torch._dynamo.config.suppress_errors = True
model = torch.compile(model, backend="eager")

# Class weights calculation (using 0-4 range since labels are already converted)
class_weights = compute_class_weight(
    'balanced', 
    classes=np.array([0, 1, 2, 3, 4]),  # Changed to 0-4
    y=[label - 1 for label in train_labels]  # Convert to 0-4 range
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

# Training configuration
loss_fn = CrossEntropyLoss(weight=class_weights)
optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
epochs = 5
batch_size = 32
grad_accumulation_steps = 2
num_training_steps = len(train_loader) * epochs
num_warmup_steps = int(0.1 * num_training_steps)
lr_scheduler = get_scheduler("linear", optimizer=optimizer, 
                           num_warmup_steps=num_warmup_steps,
                           num_training_steps=num_training_steps)

# Mixed precision setup
scaler = amp.GradScaler()

# Early stopping setup
best_loss = float('inf')
patience = 3  # Increased patience for better convergence
counter = 0

# Training loop
for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")

    # Training phase
    model.train()
    train_loss = 0
    loop = tqdm(train_loader, leave=True)

    for step, batch in enumerate(loop):
        batch = {k: v.to(device) for k, v in batch.items()}
        
        optimizer.zero_grad()

        with amp.autocast(dtype=torch.float16):
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"]  # Labels are already 0-4 from Dataset class
            )
            
            loss = outputs.loss / grad_accumulation_steps

        scaler.scale(loss).backward()

        if (step + 1) % grad_accumulation_steps == 0 or step == len(loop) - 1:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            lr_scheduler.step()

        train_loss += loss.item() * grad_accumulation_steps
        loop.set_description(f"Training Loss: {loss.item() * grad_accumulation_steps:.4f}")

    avg_train_loss = train_loss / len(train_loader)
    print(f"Training Loss: {avg_train_loss:.4f}")

    # Validation phase
    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            with amp.autocast(dtype=torch.float16):
                outputs = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    labels=batch["labels"]  # Labels are already 0-4
                )
                
                loss = outputs.loss
                logits = outputs.logits
                
                val_loss += loss.item()
                preds = torch.argmax(logits, dim=1)
                correct += (preds == batch["labels"]).sum().item()
                total += batch["labels"].size(0)

    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = correct / total
    print(f"Validation Loss: {avg_val_loss:.4f}, Accuracy: {val_accuracy:.4f}")

    # Early stopping check
    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered")
            break

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Thinh.LAPTOP-GCU8T0AJ\AppData\Local\Temp\ipykernel_18168\1262570386.py:44: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler()
c:\Users\Thinh.LAPTOP-GCU8T0AJ\.conda\envs\lugak1\lib\site-packages\torch\amp\grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(



Epoch 1/5


  0%|          | 0/771 [00:00<?, ?it/s]C:\Users\Thinh.LAPTOP-GCU8T0AJ\AppData\Local\Temp\ipykernel_18168\1262570386.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(dtype=torch.float16):
c:\Users\Thinh.LAPTOP-GCU8T0AJ\.conda\envs\lugak1\lib\site-packages\torch\amp\autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
Training Loss: 0.4293: 100%|██████████| 771/771 [42:05<00:00,  3.28s/it]
C:\Users\Thinh.LAPTOP-GCU8T0AJ\AppData\Local\Temp\ipykernel_18168\1262570386.py:100: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(dtype=torch.float16):


Training Loss: 0.8501
Validation Loss: 0.6270, Accuracy: 0.7624

Epoch 2/5


Training Loss: 0.3953: 100%|██████████| 771/771 [42:00<00:00,  3.27s/it]


...

Training Loss: 0.5594
Validation Loss: 0.6010, Accuracy: 0.7699

Epoch 4/5


Training Loss: 0.3753: 100%|██████████| 771/771 [46:53<00:00,  3.65s/it]


Training Loss: 0.5209
Validation Loss: 0.6067, Accuracy: 0.7689

Epoch 5/5


Training Loss: 0.3801: 100%|██████████| 771/771 [43:51<00:00,  3.41s/it]


Training Loss: 0.4880
Validation Loss: 0.6369, Accuracy: 0.7710


In [29]:
model.save_pretrained("mbert_star_rating_model")
tokenizer.save_pretrained("mbert_star_rating_model")


('mbert_star_rating_model\\tokenizer_config.json',
 'mbert_star_rating_model\\special_tokens_map.json',
 'mbert_star_rating_model\\vocab.txt',
 'mbert_star_rating_model\\added_tokens.json',
 'mbert_star_rating_model\\tokenizer.json')

In [19]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

all_preds = []
all_labels = []

# Đánh giá trên tập test
with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        inputs = {k: v for k, v in batch.items() if k in ["input_ids", "attention_mask", "token_type_ids"]}
        outputs = model(**inputs)
        logits = outputs.logits

        labels = batch["labels"].squeeze().to(torch.long)
        valid_indices = (labels >= 0) & (labels <= 4)
        labels = labels[valid_indices] 
        logits = logits[valid_indices]

        if labels.shape[0] == 0:
            continue

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Chuyển danh sách thành numpy array
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Báo cáo phân loại
print("\nClassification Report:")
# Ensure all classes (1-5) are included in the report and confusion matrix
print(classification_report(all_labels, all_preds, target_names=[f"{i}*" for i in range(0, 5)], labels=[0, 1, 2, 3, 4]))

# Ma trận nhầm lẫn
print("\nConfusion Matrix:")
conf_matrix = confusion_matrix(all_labels, all_preds, labels=[0, 1, 2, 3, 4])
print(conf_matrix)


Classification Report:
              precision    recall  f1-score   support

          0*       0.47      0.58      0.52       348
          1*       1.00      0.17      0.30       229
          2*       0.45      0.43      0.44       464
          3*       0.73      0.43      0.54      1143
          4*       0.83      0.95      0.89      3982

    accuracy                           0.77      6166
   macro avg       0.70      0.51      0.54      6166
weighted avg       0.77      0.77      0.75      6166


Confusion Matrix:
[[ 202    0   56   16   74]
 [  69   40   55   13   52]
 [  68    0  199   69  128]
 [  37    0   79  495  532]
 [  51    0   53   84 3794]]


In [31]:
def predict_star_rating(texts):
    model.eval()
    
    # if isinstance(texts, str):  
    #     texts = [texts]  # Convert to a list if a single string is provided
    
    # if not texts:  # Check if the input is empty
    #     raise ValueError("Input texts cannot be empty.")
    
    inputs = tokenizer(texts, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        predicted_stars = torch.argmax(probs, dim=-1).cpu().numpy() + 1  # Convert to 1-5 stars
    
    return predicted_stars

# Example predictions with multiple reviews
sample_reviews = [
    # "The product is awful, I regret buying it!",
    # "Sản phẩm quá tệ, không đáng tiền!",
    # "Not the worst, but I won’t buy it again.",
    # "Giao hàng chậm, chất lượng không như mong đợi.",
    # "It’s okay, but I expected better.",
    # "Chất lượng bình thường, không có gì đặc biệt.",
    # "Good product, but customer service needs improvement.",
    # "Sản phẩm ổn, dùng khá tốt.",
    # "Excellent quality, I highly recommend it!",
    # "Chất lượng tuyệt vời, đáng mua!"
    "Chất lượng không như mong đợi, dùng tạm được nhưng không hài lòng lắm.",
    "Sản phẩm bình thường, nhưng giá quá cao so với chất lượng.",
    "Hàng nhận được bị trầy xước nhẹ, không ảnh hưởng nhiều nhưng vẫn thất vọng.",
    "Sản phẩm tạm ổn nhưng dịch vụ chăm sóc khách hàng quá kém.",
    "Giao hàng chậm hơn dự kiến 2 tuần, trải nghiệm không tốt."
]

predictions = predict_star_rating(sample_reviews)

# Display results
for review, star in zip(sample_reviews, predictions):
    print(f"Review: {review}\nPredicted Star: {star}★\n")


Review: Chất lượng không như mong đợi, dùng tạm được nhưng không hài lòng lắm.
Predicted Star: 3★

Review: Sản phẩm bình thường, nhưng giá quá cao so với chất lượng.
Predicted Star: 4★

Review: Hàng nhận được bị trầy xước nhẹ, không ảnh hưởng nhiều nhưng vẫn thất vọng.
Predicted Star: 1★

Review: Sản phẩm tạm ổn nhưng dịch vụ chăm sóc khách hàng quá kém.
Predicted Star: 1★

Review: Giao hàng chậm hơn dự kiến 2 tuần, trải nghiệm không tốt.
Predicted Star: 1★

